In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("data/course_lead_scoring_2026.csv")

In [3]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [4]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [5]:
# Data prep
# Check if the missing values are presented in the features.
    # For categorical features, replace them with 'NA'
    # For numerical features, replace with with 0.0
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [6]:
categorical_columns = df.select_dtypes(include=['object', 'str']).columns.tolist()

for c in categorical_columns:
    df[c] = df[c].fillna('NA')

In [7]:
numerical_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

for c in numerical_columns:
    df[c] = df[c].fillna(0.0)

In [8]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [9]:
# Q1
df['industry'].mode()[0]

'technology'

In [10]:
# Q2
numerical_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

corr_matrix = df[numerical_columns].corr()

from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']


In [11]:
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

for col1, col2 in pairs:
    print(col1, col2, corr_matrix.loc[col1, col2])

interaction_count lead_score 0.9157457980418697
number_of_courses_viewed lead_score 0.7572042392104886
number_of_courses_viewed interaction_count 0.7216090038937035
annual_income interaction_count 0.12284235135955958


In [12]:
# Q3
from sklearn.metrics import mutual_info_score

def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)

for col in ['industry', 'location', 'lead_source', 'employment_status']:
    score = mutual_info_converted_score(df_train[col])
    print(col, round(score, 2))

industry 0.0
location 0.0
lead_source 0.03
employment_status 0.02


In [13]:
# Q4
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

model = LogisticRegression(
    solver='liblinear',
    C=1.0,
    max_iter=1000,
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
round(accuracy_score(y_val, y_pred), 2)

0.65

In [14]:
# Q5
features = ['lead_source', 'number_of_courses_viewed', 'interaction_count']

# Accuracy original
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

model = LogisticRegression(
    solver='liblinear', C=1.0, max_iter=1000, random_state=42
)
model.fit(X_train, y_train)

original_accuracy = accuracy_score(y_val, model.predict(X_val))
print('Original accuracy:', original_accuracy)

# Accuracy without each feature
for feature in features:
    train_dicts = df_train.drop(columns=[feature]).to_dict(orient='records')
    val_dicts = df_val.drop(columns=[feature]).to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(train_dicts)
    X_val = dv.transform(val_dicts)

    model = LogisticRegression(
        solver='liblinear', C=1.0, max_iter=1000, random_state=42
    )
    model.fit(X_train, y_train)

    accuracy_without = accuracy_score(y_val, model.predict(X_val))
    difference = original_accuracy - accuracy_without

    print(feature, accuracy_without, difference)

Original accuracy: 0.645
lead_source 0.642 0.0030000000000000027
number_of_courses_viewed 0.643 0.0020000000000000018
interaction_count 0.601 0.04400000000000004


In [15]:
# Q6
C_values = [0.000001, 0.00001, 0.0001, 0.001]

train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

for C in C_values:
    model = LogisticRegression(
        solver='liblinear',
        C=C,
        max_iter=1000,
        random_state=42
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    accuracy = accuracy_score(y_val, y_pred)

    print(C, round(accuracy, 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
